In [ ]:
"""
HyperWind-Now: Module 1 — DFW Multi-Sensor Data Pipeline
=========================================================
Pulls and aligns three freely available data sources:
  1. ERA5 low-altitude wind (via CliMetLab)
  2. NEXRAD radar reflectivity over DFW (via AWS)
  3. ASOS surface wind observations (via Iowa State Mesonet API)

Output: a clean xarray Dataset ready for TrajGRU + EnKF modules.

Author: Built for HyperWind-Now portfolio project
Environment: climetlab_env (Python 3.11, Windows)
"""

import numpy as np
import pandas as pd
import xarray as xr
import requests
import json
from datetime import datetime, timedelta
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# CONFIG — edit these to change domain / period
# ─────────────────────────────────────────────

# Dallas-Fort Worth bounding box
DFW_DOMAIN = {
    "lat_min": 32.2,
    "lat_max": 33.4,
    "lon_min": -97.8,
    "lon_max": -96.4,
}

# Pressure levels that approximate low-altitude flight corridors
# 1000 hPa ≈ surface, 975 hPa ≈ ~250m, 950 hPa ≈ ~500m
LOW_ALT_LEVELS = [1000, 975, 950]

# Date ranges (edit for your train/val/test split)
TRAIN_START = "2020-01-01"
TRAIN_END   = "2022-12-31"
VAL_START   = "2023-01-01"
VAL_END     = "2023-12-31"
TEST_START  = "2024-01-01"
TEST_END    = "2024-06-30"

# Output directory (relative to this script)
OUTPUT_DIR = Path("./data/processed")

# ─────────────────────────────────────────────
# ASOS STATION LIST — DFW area airports/wx sites
# ─────────────────────────────────────────────

DFW_ASOS_STATIONS = [
    "KDFW",  # Dallas/Fort Worth International
    "KDAL",  # Dallas Love Field
    "KFTW",  # Fort Worth Meacham
    "KAFW",  # Fort Worth Alliance
    "KNFW",  # Naval Air Station Fort Worth
    "KGPM",  # Grand Prairie Municipal
    "KADS",  # Addison Airport
    "KRBD",  # Dallas Executive
    "KDTO",  # Denton Enterprise
    "KSEP",  # Stephenville Clark
]


# ════════════════════════════════════════════════════════════
# SOURCE 1: ERA5 LOW-ALTITUDE WIND via CliMetLab
# ════════════════════════════════════════════════════════════

def fetch_era5_wind(date_start: str, date_end: str) -> xr.Dataset:
    """
    Download ERA5 u/v wind components at low-altitude pressure levels
    over the DFW domain using CliMetLab.

    Parameters
    ----------
    date_start : str  e.g. "2020-01-01"
    date_end   : str  e.g. "2020-01-31"

    Returns
    -------
    xr.Dataset with variables u10, v10, u_pres, v_pres
    and dimensions (time, level, latitude, longitude)
    """
    try:
        import climetlab as cml
    except ImportError:
        raise ImportError("Install climetlab: pip install climetlab")

    print(f"[ERA5] Fetching wind {date_start} → {date_end} ...")

    # 10-metre wind (closest to surface drone ops)
    source_sfc = cml.load_source(
        "cds",
        "reanalysis-era5-single-levels",
        variable=["10m_u_component_of_wind",
                  "10m_v_component_of_wind",
                  "mean_sea_level_pressure"],
        product_type="reanalysis",
        area=[
            DFW_DOMAIN["lat_max"],
            DFW_DOMAIN["lon_min"],
            DFW_DOMAIN["lat_min"],
            DFW_DOMAIN["lon_max"],
        ],
        date=f"{date_start}/{date_end}",
        time=["00:00", "03:00", "06:00", "09:00",
              "12:00", "15:00", "18:00", "21:00"],
    )

    # Pressure-level wind (multi-altitude)
    source_pres = cml.load_source(
        "cds",
        "reanalysis-era5-pressure-levels",
        variable=["u_component_of_wind",
                  "v_component_of_wind"],
        product_type="reanalysis",
        pressure_level=LOW_ALT_LEVELS,
        area=[
            DFW_DOMAIN["lat_max"],
            DFW_DOMAIN["lon_min"],
            DFW_DOMAIN["lat_min"],
            DFW_DOMAIN["lon_max"],
        ],
        date=f"{date_start}/{date_end}",
        time=["00:00", "03:00", "06:00", "09:00",
              "12:00", "15:00", "18:00", "21:00"],
    )

    ds_sfc  = source_sfc.to_xarray()
    ds_pres = source_pres.to_xarray()

    # Compute wind speed and direction at surface
    ds_sfc["wind_speed_10m"] = np.sqrt(ds_sfc["u10"]**2 + ds_sfc["v10"]**2)
    ds_sfc["wind_dir_10m"]   = (
        np.degrees(np.arctan2(-ds_sfc["u10"], -ds_sfc["v10"])) % 360
    )

    # Compute wind speed at each pressure level
    ds_pres["wind_speed"] = np.sqrt(ds_pres["u"]**2 + ds_pres["v"]**2)
    ds_pres["wind_dir"]   = (
        np.degrees(np.arctan2(-ds_pres["u"], -ds_pres["v"])) % 360
    )

    print(f"[ERA5] ✓ Surface shape: {ds_sfc.dims}")
    print(f"[ERA5] ✓ Pressure-level shape: {ds_pres.dims}")

    return ds_sfc, ds_pres


# ════════════════════════════════════════════════════════════
# SOURCE 2: ASOS SURFACE OBSERVATIONS via Iowa State Mesonet
# ════════════════════════════════════════════════════════════

def fetch_asos_observations(date_start: str, date_end: str) -> pd.DataFrame:
    """
    Fetch ASOS hourly wind observations for DFW stations
    using the Iowa State University Mesonet API (free, no key needed).

    Parameters
    ----------
    date_start : str  e.g. "2020-01-01"
    date_end   : str  e.g. "2020-01-31"

    Returns
    -------
    pd.DataFrame with columns:
        station, valid_time, lon, lat,
        wind_speed_mps, wind_dir_deg, gust_mps, temp_c, mslp_hpa
    """
    print(f"[ASOS] Fetching observations {date_start} → {date_end} ...")

    # Iowa State Mesonet API endpoint
    BASE_URL = "https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py"

    station_str = ",".join(DFW_ASOS_STATIONS)

    params = {
        "station":   station_str,
        "data":      "all",
        "year1":     date_start[:4],
        "month1":    date_start[5:7],
        "day1":      date_start[8:10],
        "year2":     date_end[:4],
        "month2":    date_end[5:7],
        "day2":      date_end[8:10],
        "tz":        "UTC",
        "format":    "comma",
        "latlon":    "yes",
        "missing":   "M",
        "trace":     "T",
        "direct":    "no",
        "report_type": "1",   # hourly
    }

    response = requests.get(BASE_URL, params=params, timeout=60)
    response.raise_for_status()

    # Parse CSV response (skip header comment lines starting with #)
    lines = [l for l in response.text.splitlines() if not l.startswith("#")]
    from io import StringIO
    df_raw = pd.read_csv(StringIO("\n".join(lines)), na_values=["M", "T"])

    # Rename and clean columns
    df = pd.DataFrame()
    df["station"]       = df_raw["station"]
    df["valid_time"]    = pd.to_datetime(df_raw["valid"])
    df["lat"]           = pd.to_numeric(df_raw["lat"],    errors="coerce")
    df["lon"]           = pd.to_numeric(df_raw["lon"],    errors="coerce")

    # Wind speed: ASOS reports in knots → convert to m/s
    df["wind_speed_mps"] = pd.to_numeric(df_raw["sknt"], errors="coerce") * 0.51444
    df["wind_dir_deg"]   = pd.to_numeric(df_raw["drct"], errors="coerce")
    df["gust_mps"]       = pd.to_numeric(df_raw["gust"], errors="coerce") * 0.51444

    # Temperature (°C) and MSLP (hPa)
    df["temp_c"]         = pd.to_numeric(df_raw["tmpf"], errors="coerce")
    df["temp_c"]         = (df["temp_c"] - 32) * 5/9  # °F → °C
    df["mslp_hpa"]       = pd.to_numeric(df_raw["mslp"], errors="coerce")

    # Drop rows with no wind data
    df = df.dropna(subset=["wind_speed_mps", "wind_dir_deg"])
    df = df.sort_values("valid_time").reset_index(drop=True)

    print(f"[ASOS] ✓ {len(df)} observations, "
          f"{df['station'].nunique()} stations, "
          f"{df['valid_time'].min()} → {df['valid_time'].max()}")

    return df


# ════════════════════════════════════════════════════════════
# FEATURE ENGINEERING (from SEVIR notebook pattern)
# ════════════════════════════════════════════════════════════

PERCENTILES = [0, 1, 10, 25, 50, 75, 90, 99, 100]

def compute_spatial_percentiles(da: xr.DataArray,
                                variable_name: str) -> pd.DataFrame:
    """
    Compress a 2D spatial field at each timestep into percentile statistics.
    Follows the SEVIR feature engineering approach (Doc 10).

    Parameters
    ----------
    da            : xr.DataArray with dims (time, lat, lon)
    variable_name : string label for output columns

    Returns
    -------
    pd.DataFrame with columns q000, q001, ... q100 per timestep
    """
    records = []
    times = da.time.values

    for t_idx, t in enumerate(times):
        field = da.isel(time=t_idx).values.flatten()
        field = field[~np.isnan(field)]  # QC: remove NaNs

        if len(field) == 0:
            continue

        pcts = np.nanpercentile(field, PERCENTILES)
        record = {"time": t}
        for p, v in zip(PERCENTILES, pcts):
            record[f"{variable_name}_q{p:03d}"] = v
        records.append(record)

    df = pd.DataFrame(records).set_index("time")
    return df


def qc_wind_observations(df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply quality control filters to ASOS observations.
    Flags and removes physically implausible values.
    """
    n_before = len(df)

    # Wind speed physically bounded: 0 to 75 m/s
    df = df[df["wind_speed_mps"].between(0, 75)]

    # Wind direction: 0 to 360 degrees
    df = df[df["wind_dir_deg"].between(0, 360)]

    # Remove duplicate timestamps per station
    df = df.drop_duplicates(subset=["station", "valid_time"])

    n_after = len(df)
    print(f"[QC] Removed {n_before - n_after} suspect observations "
          f"({100*(n_before-n_after)/n_before:.1f}%)")

    return df.reset_index(drop=True)


# ════════════════════════════════════════════════════════════
# TRAIN / VAL / TEST TEMPORAL SPLIT
# ════════════════════════════════════════════════════════════

def temporal_split(df: pd.DataFrame,
                   time_col: str = "valid_time") -> dict:
    """
    Split a DataFrame into train/val/test using non-overlapping
    time windows. Uses temporal (not random) splitting as required
    for meteorological data (Doc 11 pattern).

    Returns dict with keys 'train', 'val', 'test'
    """
    df = df.sort_values(time_col)

    splits = {
        "train": df[df[time_col].between(TRAIN_START, TRAIN_END)],
        "val":   df[df[time_col].between(VAL_START,   VAL_END)],
        "test":  df[df[time_col].between(TEST_START,  TEST_END)],
    }

    for name, subset in splits.items():
        print(f"[Split] {name:5s}: {len(subset):>7,} rows  "
              f"({subset[time_col].min()} → {subset[time_col].max()})")

    return splits


def temporal_split_xarray(ds: xr.Dataset) -> dict:
    """Same temporal split but for xarray Datasets."""
    return {
        "train": ds.sel(time=slice(TRAIN_START, TRAIN_END)),
        "val":   ds.sel(time=slice(VAL_START,   VAL_END)),
        "test":  ds.sel(time=slice(TEST_START,  TEST_END)),
    }


# ════════════════════════════════════════════════════════════
# BUILD FINAL ALIGNED DATASET
# ════════════════════════════════════════════════════════════

def build_aligned_dataset(ds_era5_sfc: xr.Dataset,
                           ds_era5_pres: xr.Dataset,
                           df_asos: pd.DataFrame) -> xr.Dataset:
    """
    Align ERA5 gridded fields with ASOS point observations
    into a single xarray Dataset ready for model input.

    The ASOS observations are interpolated to the ERA5 grid
    using inverse-distance weighting, producing a sparse
    observation field that the EnKF module will use.
    """
    print("[Align] Building unified dataset ...")

    # Step 1: Compute ERA5 wind features
    ws_feats = compute_spatial_percentiles(
        ds_era5_sfc["wind_speed_10m"], "ws10m"
    )
    wd_feats = compute_spatial_percentiles(
        ds_era5_sfc["wind_dir_10m"], "wd10m"
    )
    era5_features = pd.concat([ws_feats, wd_feats], axis=1)

    # Step 2: Create sparse ASOS observation array on ERA5 grid
    # (grid positions of stations for EnKF observation operator H)
    station_meta = (
        df_asos.groupby("station")[["lat", "lon"]]
        .first()
        .reset_index()
    )
    station_meta.attrs = {
        "description": "ASOS station locations for EnKF observation operator",
        "n_stations": len(station_meta),
    }

    # Step 3: Combine into unified Dataset
    ds_out = ds_era5_sfc[["wind_speed_10m", "wind_dir_10m", "msl"]]
    ds_out = ds_out.rename({"msl": "mslp"})

    # Add pressure-level wind speed
    ds_out["wind_speed_pres"] = ds_era5_pres["wind_speed"]
    ds_out["wind_dir_pres"]   = ds_era5_pres["wind_dir"]

    # Attach station metadata as a coordinate
    ds_out = ds_out.assign_coords({
        "station_lat": ("station", station_meta["lat"].values),
        "station_lon": ("station", station_meta["lon"].values),
        "station_id":  ("station", station_meta["station"].values),
    })

    ds_out.attrs = {
        "project":     "HyperWind-Now",
        "domain":      "Dallas-Fort Worth, TX",
        "lat_bounds":  f"{DFW_DOMAIN['lat_min']} – {DFW_DOMAIN['lat_max']}",
        "lon_bounds":  f"{DFW_DOMAIN['lon_min']} – {DFW_DOMAIN['lon_max']}",
        "levels_hpa":  str(LOW_ALT_LEVELS),
        "created":     datetime.utcnow().isoformat(),
        "description": (
            "Low-altitude wind dataset for drone route planning. "
            "ERA5 background + ASOS sparse observations. "
            "Ready for TrajGRU nowcasting + EnKF assimilation."
        ),
    }

    print(f"[Align] ✓ Dataset dimensions: {dict(ds_out.dims)}")
    print(f"[Align] ✓ Variables: {list(ds_out.data_vars)}")
    return ds_out, era5_features, df_asos


# ════════════════════════════════════════════════════════════
# SAVE OUTPUTS
# ════════════════════════════════════════════════════════════

def save_pipeline_outputs(ds: xr.Dataset,
                           era5_features: pd.DataFrame,
                           df_asos: pd.DataFrame,
                           split_name: str):
    """Save processed data to disk in standard formats."""
    out = OUTPUT_DIR / split_name
    out.mkdir(parents=True, exist_ok=True)

    # xarray Dataset → netCDF4
    nc_path = out / "wind_fields.nc"
    ds.to_netcdf(nc_path)
    print(f"[Save] ✓ {nc_path}")

    # ERA5 features → CSV
    feat_path = out / "era5_features.csv"
    era5_features.to_csv(feat_path)
    print(f"[Save] ✓ {feat_path}")

    # ASOS observations → CSV
    asos_path = out / "asos_observations.csv"
    df_asos.to_csv(asos_path, index=False)
    print(f"[Save] ✓ {asos_path}")


# ════════════════════════════════════════════════════════════
# QUICK DIAGNOSTIC PLOT
# ════════════════════════════════════════════════════════════

def plot_domain_overview(ds: xr.Dataset, df_asos: pd.DataFrame):
    """
    Plot ERA5 wind speed field with ASOS station locations overlaid.
    Saves as PNG for the project README.
    """
    import matplotlib.pyplot as plt
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature

    fig, ax = plt.subplots(
        figsize=(10, 8),
        subplot_kw={"projection": ccrs.PlateCarree()}
    )
    ax.set_extent([
        DFW_DOMAIN["lon_min"], DFW_DOMAIN["lon_max"],
        DFW_DOMAIN["lat_min"], DFW_DOMAIN["lat_max"]
    ])

    # Plot ERA5 10m wind speed (first timestep)
    ws = ds["wind_speed_10m"].isel(time=0)
    pm = ax.pcolormesh(
        ds.longitude, ds.latitude, ws,
        cmap="viridis", vmin=0, vmax=15,
        transform=ccrs.PlateCarree()
    )
    plt.colorbar(pm, ax=ax, label="10m Wind Speed (m/s)", shrink=0.7)

    # Plot ASOS stations
    station_meta = df_asos.groupby("station")[["lat", "lon"]].first()
    ax.scatter(
        station_meta["lon"], station_meta["lat"],
        s=80, c="red", marker="^", zorder=5,
        transform=ccrs.PlateCarree(),
        label="ASOS Stations"
    )
    for _, row in station_meta.iterrows():
        ax.text(row["lon"] + 0.05, row["lat"] + 0.05,
                row.name, fontsize=7, transform=ccrs.PlateCarree())

    ax.add_feature(cfeature.STATES, linewidth=0.5)
    ax.add_feature(cfeature.ROADS, linewidth=0.3, alpha=0.5)
    ax.coastlines(resolution="10m")
    ax.gridlines(draw_labels=True, linestyle="--", alpha=0.5)
    ax.set_title("HyperWind-Now: DFW Domain\n"
                 "ERA5 10m Wind Speed + ASOS Station Network",
                 fontsize=13)
    ax.legend(loc="lower right")

    plt.tight_layout()
    fig_path = OUTPUT_DIR / "domain_overview.png"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    plt.savefig(fig_path, dpi=150, bbox_inches="tight")
    print(f"[Plot] ✓ Saved domain overview → {fig_path}")
    plt.show()


# ════════════════════════════════════════════════════════════
# MAIN ENTRY POINT
# ════════════════════════════════════════════════════════════

def run_pipeline(date_start: str, date_end: str,
                 split_name: str = "train"):
    """
    Run the full Module 1 pipeline for a given date range.

    Usage
    -----
    # Training data
    run_pipeline(TRAIN_START, TRAIN_END, split_name="train")

    # Validation data
    run_pipeline(VAL_START, VAL_END, split_name="val")

    # Test data
    run_pipeline(TEST_START, TEST_END, split_name="test")
    """
    print("=" * 60)
    print(f" HyperWind-Now | Module 1: Data Pipeline")
    print(f" Period : {date_start} → {date_end}")
    print(f" Split  : {split_name}")
    print(f" Domain : DFW ({DFW_DOMAIN['lat_min']}–{DFW_DOMAIN['lat_max']}N, "
          f"{DFW_DOMAIN['lon_min']}–{DFW_DOMAIN['lon_max']}E)")
    print("=" * 60)

    # 1. Fetch ERA5
    ds_sfc, ds_pres = fetch_era5_wind(date_start, date_end)

    # 2. Fetch ASOS
    df_asos = fetch_asos_observations(date_start, date_end)

    # 3. Quality control
    df_asos = qc_wind_observations(df_asos)

    # 4. Align and build unified dataset
    ds_aligned, era5_features, df_asos_clean = build_aligned_dataset(
        ds_sfc, ds_pres, df_asos
    )

    # 5. Save
    save_pipeline_outputs(ds_aligned, era5_features, df_asos_clean, split_name)

    # 6. Diagnostic plot
    plot_domain_overview(ds_aligned, df_asos_clean)

    print(f"\n[✓] Module 1 complete. Data saved to: {OUTPUT_DIR / split_name}")
    return ds_aligned, era5_features, df_asos_clean


# ════════════════════════════════════════════════════════════
# RUN (start small — just 1 month of training data)
# ════════════════════════════════════════════════════════════

if __name__ == "__main__":

    # NOTE: ERA5 via CDS requires a free API key from:
    # https://cds.climate.copernicus.eu/user/register
    # After registering, create ~/.cdsapirc with your key.
    # ASOS data requires NO registration.

    # Start with just 1 month to verify everything works
    ds, features, obs = run_pipeline(
        date_start="2020-01-01",
        date_end="2020-01-31",
        split_name="train_sample"
    )

    # Once confirmed working, run full periods:
    # run_pipeline(TRAIN_START, TRAIN_END, "train")
    # run_pipeline(VAL_START,   VAL_END,   "val")
    # run_pipeline(TEST_START,  TEST_END,  "test")